# Análise de Gênero\n\nEvolução da participação de medalhistas por gênero ao longo das edições.

In [1]:
import pandas as pd
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

In [2]:
BRONZE_DIR = Path("../../bronze")
OUTPUT_DIR = Path(".")

df_sexo = pd.read_parquet(BRONZE_DIR / "atletas_por_sexo.parquet")

genero_ano = df_sexo.pivot_table(
    index="ano",
    columns="genero",
    values="atleta",
    aggfunc="count",
    fill_value=0
).reset_index()
genero_ano.columns.name = None

for col in ["Male", "Female"]:
    if col not in genero_ano.columns:
        genero_ano[col] = 0

genero_ano["Total"] = genero_ano.select_dtypes(include="number").sum(axis=1)
genero_ano["Percentual_Feminino"] = (genero_ano["Female"] / genero_ano["Total"] * 100).round(1)

genero_ano.to_csv(OUTPUT_DIR / "genero_summary.csv", index=False)

print(f"Edicoes analisadas: {len(genero_ano)}")
genero_ano.tail(10)

Edicoes analisadas: 38


,ano,Female,Male,Mixed,Total,Percentual_Feminino
28,2006,228,289,18,2541,9.0
29,2008,920,1094,66,4088,22.5
30,2010,231,285,18,2544,9.1
31,2012,916,1010,57,3995,22.9
32,2014,246,300,69,2629,9.4
33,2016,962,1038,63,4079,23.6
34,2018,256,294,123,2691,9.5
35,2020,1113,1132,189,4454,25.0
36,2022,265,297,164,2748,9.6
37,2024,1162,1153,0,4339,26.8


In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

genero_ano.plot(x="ano", y=["Male", "Female"], kind="area", stacked=True,
                color=["#4A90D9", "#E74C6F"], alpha=0.7, ax=ax1)
ax1.set_title("Medalhistas por Gênero ao Longo do Tempo")
ax1.set_xlabel("Ano")
ax1.set_ylabel("Quantidade")

ax2.plot(genero_ano["ano"], genero_ano["Percentual_Feminino"], color="#E74C6F", linewidth=2, marker="o", markersize=3)
ax2.set_title("Percentual de Medalhistas Femininas (%)")
ax2.set_xlabel("Ano")
ax2.set_ylabel("Percentual (%)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "genero_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [4]:
metadata = {
    "nome": "Análise de Gênero",
    "fonte": "Camada bronze - atletas_por_sexo",
    "descricao": "Evolução da participação por gênero nas olimpíadas",
    "formato": "csv/png",
    "colunas": [{"nome": col, "tipo": str(genero_ano[col].dtype)} for col in genero_ano.columns],
    "quantidade_linhas": len(genero_ano),
    "data_coleta": datetime.now().strftime("%Y-%m-%d"),
    "observacoes": ""
}

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)